# GRU4Rec — training & validation (Colab)

Trening baseline **GRU4Rec** z **PyTorch Lightning** na danych Yoochoose.

**Logowane metryki (val):**
- `val/loss`, `val/perplexity`
- `val/hit@1`, `val/recall@5`, `val/recall@10`, `val/recall@20`
- `val/mrr@20`, `val/ndcg@20`
- `val/recall@5_pop`, `val/recall@10_pop`, `val/recall@20_pop` (baseline popularności)

Checkpointy: `checkpoints/gru4rec/<run_name>/` na Google Drive.
W&B: `project-nn/adm-project-tgnn` (stałe w `src/config/wandb_settings.py`).

## 0. Konfiguracja użytkownika

In [1]:
!pip install -q lightning wandb pandas pyarrow tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 44.9 MB/s eta 0:00:00


In [2]:
from pathlib import Path

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/adm-project-tgnn")
REPO_DIR = Path("/content/adm-project-tgnn")
REPO_URL = "https://github.com/SwankyBombul/adm-project-tgnn"

PROCESSED_VARIANT = "subsample_1_32_clicks_only"
RUN_NAME = "gru4rec-baseline"

# Hyperparametry treningu
NUM_EPOCHS = 1
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
EMBEDDING_DIM = 128
HIDDEN_DIM = 128

## 1. Repozytorium i zależności

In [7]:
import subprocess
import sys

if REPO_URL and not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

if not REPO_DIR.is_dir():
    raise FileNotFoundError(f"Repo not found at {REPO_DIR}")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_DIR)],
    check=True,
)
subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

print(f"Repo ready: {REPO_DIR}")

Repo ready: /content/adm-project-tgnn


## 2. Google Drive — mount, walidacja, rozpakowanie danych

In [4]:
from pprint import pprint

from src.colab.setup import prepare_colab_session
from src.config.train_config import TrainConfig

# prepare_colab_session: mount Drive -> validate paths -> unzip processed.zip
config = TrainConfig.for_colab(
    DRIVE_PROJECT_DIR,
    model_name="gru4rec",
    processed_variant=PROCESSED_VARIANT,
    run_name=RUN_NAME,
    wandb_run_name=RUN_NAME,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
)
config = prepare_colab_session(config)

pprint(
    {
        "processed_dir": config.processed_dir,
        "checkpoint_dir": config.checkpoint_dir,
        "num_epochs": config.num_epochs,
        "batch_size": config.batch_size,
    }
)

Mounted at /content/drive
{'batch_size': 256,
 'checkpoint_dir': PosixPath('/content/drive/MyDrive/adm-project-tgnn/checkpoints/gru4rec/gru4rec-baseline'),
 'num_epochs': 1,
 'processed_dir': PosixPath('/content/adm-project-tgnn/data/processed/subsample_1_32_clicks_only')}


## 3. Weights & Biases

In [5]:
from src.config.wandb_settings import WANDB_ENTITY, WANDB_PROJECT, verify_wandb_access

wandb_info = verify_wandb_access()
print(f"wandb target: {WANDB_ENTITY}/{WANDB_PROJECT}")
for key, value in wandb_info.items():
    print(f"  {key}: {value}")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: koostosh (ADM-lists) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb target: project-nn/adm-project-tgnn
  entity: project-nn
  project: adm-project-tgnn
  viewer: koostosh
  project_path: project-nn/adm-project-tgnn
  available_entities: ['ADM-lists', 'VXNlcjoyOTI3OTIy', 'koostosh']


## 4. Trening GRU4Rec (Lightning)

Po `git pull` przeładuj moduły przed treningiem (Colab trzyma starą wersję w pamięci).

In [12]:
import importlib

import src.data.gru4rec
import src.training.train_gru4rec as train_module

importlib.reload(src.data.gru4rec)
importlib.reload(train_module)

from src.training.train_gru4rec import train_gru4rec

train_gru4rec(config)
print(f"\nTraining finished. Checkpoints: {config.checkpoint_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


ValueError: Caught ValueError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 57, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/content/adm-project-tgnn/src/data/gru4rec.py", line 43, in gru4rec_collate_fn
    def gru4rec_collate_fn(
               ^^^^^^^
ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


## 5. Podsumowanie

Po treningu sprawdź run w [wandb](https://wandb.ai) (`project-nn/adm-project-tgnn`) oraz pliki `epoch_*.ckpt` na Drive.

**Następny krok:** ewaluacja na `test_internal` / `challenge_test` — do dopisania osobno.